<a href="https://colab.research.google.com/github/nabillahumi/S5-Mesin_Learning/blob/main/Jobsheet%2014/Praktikum%201%20(JS14).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Langkah 1 - Import Library

In [8]:
import os
import zipfile
import numpy as np
from PIL import Image

print("1. Membuat struktur folder dataset...")
os.makedirs('dataset/training_set/cats', exist_ok=True)
os.makedirs('dataset/training_set/dogs', exist_ok=True)
os.makedirs('dataset/test_set/cats', exist_ok=True)
os.makedirs('dataset/test_set/dogs', exist_ok=True)
os.makedirs('dataset/single_prediction', exist_ok=True)

# Fungsi untuk membuat gambar dummy (agar format file zip valid dan bisa dilatih)
def create_dummy_image(path):
    img_array = np.random.randint(0, 256, (64, 64, 3), dtype=np.uint8)
    img = Image.fromarray(img_array)
    img.save(path)

print("2. Menyiapkan file gambar sampel di dalam folder...")
for i in range(5):
    create_dummy_image(f'dataset/training_set/cats/cat_{i}.jpg')
    create_dummy_image(f'dataset/training_set/dogs/dog_{i}.jpg')
    create_dummy_image(f'dataset/test_set/cats/cat_test_{i}.jpg')
    create_dummy_image(f'dataset/test_set/dogs/dog_test_{i}.jpg')

create_dummy_image('dataset/single_prediction/cat_or_dog_1.jpg')

print("3. Mengompres folder menjadi dataset.zip yang valid...")
with zipfile.ZipFile('dataset.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk('dataset'):
        for file in files:
            file_path = os.path.join(root, file)
            zipf.write(file_path, arcname=file_path)

print("4. Mengekstrak kembali dataset.zip...")
with zipfile.ZipFile('dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print("Yeay! File dataset.zip berhasil dibuat, diekstrak, dan siap digunakan!")

1. Membuat struktur folder dataset...
2. Menyiapkan file gambar sampel di dalam folder...
3. Mengompres folder menjadi dataset.zip yang valid...
4. Mengekstrak kembali dataset.zip...
Yeay! File dataset.zip berhasil dibuat, diekstrak, dan siap digunakan!


In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

#Langkah 2 - Pra Pengolahan Data

##Langkah 2.1. Pra Pengolahan Data Training

In [9]:
train_datagen = ImageDataGenerator(rescale = 1./255,
	                                   shear_range = 0.2,
	                                   zoom_range = 0.2,
	                                   horizontal_flip = True)
training_set = train_datagen.flow_from_directory('dataset/training_set',
	                                                 target_size = (64, 64),
	                                                 batch_size = 32,
	                                                 class_mode = 'binary')

Found 10 images belonging to 2 classes.


##Langkah 2.2. Pra Pengolahan Data Testing

In [10]:
	test_datagen = ImageDataGenerator(rescale = 1./255)
	test_set = test_datagen.flow_from_directory('dataset/test_set',
	                                            target_size = (64, 64),
	                                            batch_size = 32,
	                                            class_mode = 'binary')

Found 10 images belonging to 2 classes.


#Langkah 3 - Pembuatan Model CNN

##Langkah 3.1.  - Inisiasi Model CNN

In [11]:
cnn = tf.keras.models.Sequential()

##Langkah 3.2. - Pembuatan Layer Konvolusi 1

In [12]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64, 64, 3]))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


#L#angkah 3.3 - Pembuatan Layer Pooling 1

In [13]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

##Langkah 3.4 - Pembuatan Layer Konvolusi 2 dan Pooling 2

In [15]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

##Langkah 3.5 - Flattening

In [16]:
cnn.add(tf.keras.layers.Flatten())

##Langkah 3.6 - Fully Connected Layer 1 (Input)

In [17]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

##Langkah 3.7 - Fully Connected Layer 2 (Output)

In [18]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

##Langkah 3.8 - Compile Model CNN

In [19]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

#Langkah 4 - Fit CNN

In [20]:
cnn.fit(x = training_set, validation_data = test_set, epochs = 25)

Epoch 1/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.5000 - loss: 0.6959 - val_accuracy: 0.5000 - val_loss: 0.6962
Epoch 2/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step - accuracy: 0.5000 - loss: 0.6962 - val_accuracy: 0.5000 - val_loss: 0.6956
Epoch 3/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step - accuracy: 0.5000 - loss: 0.6938 - val_accuracy: 0.5000 - val_loss: 0.6972
Epoch 4/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step - accuracy: 0.5000 - loss: 0.6931 - val_accuracy: 0.5000 - val_loss: 0.6977
Epoch 5/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 333ms/step - accuracy: 0.5000 - loss: 0.6939 - val_accuracy: 0.5000 - val_loss: 0.6953
Epoch 6/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 261ms/step - accuracy: 0.5000 - loss: 0.6918 - val_accuracy: 0.5000 - val_loss: 0.6948
Epoch 7/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 299ms/step - accuracy: 0.5000 - loss: 0.6917 - val_accuracy: 0.5000 - val_loss: 0.6952
Epoch 8/25
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 256ms/step - accuracy: 0.5000 - loss: 0.6907 - val_accuracy: 0.3000 - val_loss: 0.

#Langkah 5 - Prediksi dengan 1 Citra

In [25]:
import numpy as np
from keras.preprocessing import image
test_image = image.load_img('dataset/single_prediction/cat_or_dog_1.jpg', target_size = (64, 64))
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis = 0)
result = cnn.predict(test_image)
training_set.class_indices
if result[0][0] == 1:
  printprediction = 'dog'
else:
  prediction = 'cat'

# Menampilkan hasil sesuai permintaan
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
dog
